# 🏭 Industrial AI: Predictive Maintenance & OEE Prognostics Pipeline
This notebook demonstrates the end-to-end Machine Learning and Feature Engineering pipeline used to model **Remaining Useful Life (RUL)** of turbofan engines using the **NASA CMAPSS dataset**, linking predictions directly to manufacturing **Overall Equipment Effectiveness (OEE)** KPIs.

## 🛠️ 1. Environment Setup & Imports
We import standard scientific computing packages along with our modular pipeline modules.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path to load project source packages
sys.path.append(os.path.abspath('..'))
from src.config import Config
from src.pipeline.data_loader import load_dataset
from src.pipeline.feature_engineering import FeatureEngineer

## 📊 2. Data Loading & Exploration
We load the CMAPSS training dataset. It contains multi-sensor telemetry readings logged cycle-by-cycle across multiple engine units until they reach functional failure.

In [ ]:
# Check dataset existence
data_path = "../data/train_FD001.txt"
if not os.path.exists(data_path):
    # Fallback to simulated large CSV if standard CMAPSS is not local
    data_path = "../data/fleet_telemetry_large.csv"

df = load_dataset(data_path)
print(f"Loaded dataset from: {data_path}")
print(f"Shape: {df.shape}")
df.head()

## ⚙️ 3. Feature Engineering & Target Generation
For predictive maintenance, we compute a target label called **Remaining Useful Life (RUL)** by finding the maximum cycle recorded for each engine and calculating `RUL = max_cycle - current_cycle`.

We also compute rolling window statistics (mean and standard deviation) per engine over a sliding timeline to capture degradation trends and smooth out sensor noise.

In [ ]:
fe = FeatureEngineer(window_size=10)

# 1. Calculate Target RUL
df_with_rul = fe.add_rul(df)

# 2. Vectorized Rolling Features
df_engineered = fe.compute_rolling_features(df_with_rul)

print(f"Engineered feature space shape: {df_engineered.shape}")
df_engineered[['engine_id', 'cycle', 'RUL', 'sensor_11', 'sensor_11_roll_mean', 'sensor_4', 'sensor_4_roll_mean']].head(10)

## 📈 4. Visualizing Sensor Degradation
We select a single engine unit (e.g. Engine #1) and plot its sensor trends over its operational cycle count to visualize wear degradation.

In [ ]:
# Filter for a single engine
engine_1 = df_engineered[df_engineered['engine_id'] == 1].sort_values(by='cycle')

plt.figure(figsize=(14, 8))

# Plot Sensor 11 (Core Temperature) trend
plt.subplot(2, 1, 1)
plt.plot(engine_1['cycle'], engine_1['sensor_11'], label='Raw Sensor 11', color='lightblue', alpha=0.7)
plt.plot(engine_1['cycle'], engine_1['sensor_11_roll_mean'], label='Rolling Mean (window=10)', color='blue', linewidth=2)
plt.title('Sensor 11 (Turbine Inlet Temperature) Degradation - Engine #1')
plt.xlabel('Cycle')
plt.ylabel('Temperature')
plt.legend()
plt.grid(True, linestyle='--')

# Plot Target RUL
plt.subplot(2, 1, 2)
plt.plot(engine_1['cycle'], engine_1['RUL'], label='Remaining Useful Life (RUL)', color='red', linewidth=2)
plt.title('Remaining Useful Life Timeline')
plt.xlabel('Cycle')
plt.ylabel('RUL (cycles)')
plt.legend()
plt.grid(True, linestyle='--')

plt.tight_layout()
plt.show()

## 🧠 5. Model Training & Evaluation
We split the engineered dataset and train a constrained `RandomForestRegressor` (`max_depth=15`, `min_samples_leaf=5`) to predict the RUL based on scaled features.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Fit scaler and retrieve train features
X, y = fe.fit_transform(df)

# Train/Validation Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=Config.TEST_SIZE, random_state=Config.RANDOM_STATE
)

# Fit RandomForest model
model = RandomForestRegressor(
    n_estimators=Config.N_ESTIMATORS,
    max_depth=15,
    min_samples_leaf=5,
    random_state=Config.RANDOM_STATE,
    n_jobs=-1
)
model.fit(X_train, y_train)

# Evaluate
preds = model.predict(X_val)
print(f"Validation RMSE: {np.sqrt(mean_squared_error(y_val, preds)):.2f} cycles")
print(f"Validation MAE:  {mean_absolute_error(y_val, preds):.2f} cycles")
print(f"Validation R2:   {r2_score(y_val, preds):.4f}")

## 📊 6. Feature Importances
We plot the top wear-driving features ranked by the Random Forest model to understand what sensors drive machinery failure warnings.

In [ ]:
importances = model.feature_importances_
feat_importances = pd.Series(importances, index=X.columns)
feat_importances.nlargest(10).plot(kind='barh', color='darkblue')
plt.title('Top 10 Feature Importances')
plt.xlabel('Relative Importance')
plt.ylabel('Feature')
plt.show()

## 🏭 7. Linking RUL Predictions to Manufacturing OEE
Overall Equipment Effectiveness (OEE) represents productivity. We map the predicted RUL to OEE factors:
1. **Availability**: Degrades if predicted RUL falls below a critical threshold of 30 cycles.
2. **Performance**: Measures cycle usage relative to baseline lifecycles.
3. **Quality**: Drops if sensor core temperatures (Sensor 11) overheat.

Here, we simulate this calculation for Engine #1.

In [ ]:
# Run prediction on Engine #1 history
X_eng1 = fe.transform(df[df['engine_id'] == 1])
pred_ruls = model.predict(X_eng1)

engine_1_analysis = df[df['engine_id'] == 1][['cycle', 'sensor_11']].copy()
engine_1_analysis['predicted_RUL'] = pred_ruls

# Calculate OEE components
engine_1_analysis['Availability'] = engine_1_analysis['predicted_RUL'].apply(
    lambda r: 1.0 if r >= 30.0 else max(0.0, r / 30.0)
)
engine_1_analysis['Performance'] = engine_1_analysis['cycle'].apply(
    lambda c: min(1.0, c / 300.0)
)
engine_1_analysis['Quality'] = engine_1_analysis['sensor_11'].apply(
    lambda s: 1.0 if s <= 480.0 else max(0.0, 1.0 - (s - 480.0) / (525.0 - 480.0))
)
engine_1_analysis['OEE'] = (
    engine_1_analysis['Availability'] * 
    engine_1_analysis['Performance'] * 
    engine_1_analysis['Quality']
)

# Plot OEE decline towards failure point
plt.figure(figsize=(12, 6))
plt.plot(engine_1_analysis['cycle'], engine_1_analysis['OEE'], label='Overall OEE', color='purple', linewidth=3)
plt.plot(engine_1_analysis['cycle'], engine_1_analysis['Availability'], label='Availability Factor', linestyle='--', color='red')
plt.plot(engine_1_analysis['cycle'], engine_1_analysis['Quality'], label='Quality Factor', linestyle=':', color='green')
plt.title('Simulated OEE Metrics over Engine Lifecycle - Engine #1')
plt.xlabel('Cycle count')
plt.ylabel('Score (0 to 1)')
plt.legend()
plt.grid(True, linestyle=':')
plt.show()